In [ ]:
import os
from src.arguments import ModelArguments, DataArguments
from src.model.model import MMEBModel
from src.model.processor import load_processor, QWEN2_VL, VLM_IMAGE_TOKENS, \
    Qwen2_VL_process_fn, LLAVA_QWEN2, FastVLM_process_fn
from src.utils import batch_to_device
from PIL import Image
import numpy as np
from src.model.llava.model import LlavaQwen2ForCausalLM
import torch
import math
%matplotlib inline
import matplotlib.pyplot as plt
import torch.nn.functional as F

from transformers.image_transforms import (
    convert_to_rgb,
    resize,
)

In [ ]:
# model_args = ModelArguments(
#     model_name='raghavlite/B3_Qwen2_2B',
#     pooling='last',
#     normalize=True,
#     model_backbone='qwen2_vl',
#     lora=True
# )
model_args = ModelArguments(
    model_name='apple/FastVLM-0.5B',
    pooling='last',
    normalize=True,
    model_backbone=LLAVA_QWEN2,
    modality_gated_pooling = True,
    lora=True,
)
data_args = DataArguments()

processor = load_processor(model_args, None)
model = MMEBModel.build(model_args)
model = model.to('cuda', dtype=torch.bfloat16)
model.eval()

In [ ]:
processor_inputs = {
    "text": [f'{VLM_IMAGE_TOKENS[LLAVA_QWEN2]} Represent the given image with the following question: What is in the image',
          f'{VLM_IMAGE_TOKENS[LLAVA_QWEN2]} Represent the given image with the following question: What is in the image'],
    "images": [Image.open('example.jpg').resize((500, 1025)),
            Image.open('example.jpg')],
}

inputs = FastVLM_process_fn(
    processor_inputs,
    processor, 
    # square_padding=True
    )
inputs = batch_to_device(inputs, "cuda")
inputs['images'][0].shape, inputs['images'][1].shape



In [ ]:
input_ids = inputs["input_ids"]
attention_mask = inputs["attention_mask"]

eos_id = processor.tokenizer.eos_token_id
last_idx = attention_mask.long().sum(dim=1) - 1
last_ids = input_ids[torch.arange(input_ids.size(0)), last_idx]

print("eos_id:", eos_id)
print("last_ids:", last_ids.tolist())
print("all end with eos:", bool((last_ids == eos_id).all()))

In [ ]:
type(model.encoder.get_vision_tower().vision_tower.model)

In [ ]:
output = model.encode_input(inputs)
output

In [ ]:
x = torch.randn(1, 3, 768, 768).to('cuda', dtype=torch.bfloat16)
with torch.no_grad():
    y = model.encoder.get_vision_tower()(x)
y.shape